In [1]:
import torch  # 导入 PyTorch 核心计算库
import torch.optim as optim  # 导入优化器模块
from torch_geometric.loader import DataLoader  # 导入 PyG 图数据加载器
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score  # 导入分类指标评估工具
import numpy as np  # 导入科学计算库
import os  # 导入文件路径管理库
import random  # 导入随机数控制
from datetime import datetime  # 导入时间日期管理
from tqdm import tqdm  # 导入命令行进度条显示工具
import sys

In [2]:
# 从 models 子包中导入所有待测试的图神经网络架构
from models.test_models import PureSAGEModel, PureDynamicGATModel, PureGCNModel, PureClassicGATModel
from models.hybrid_parallel_model import ParallelHybridStressModel 
from loss import StressTaskLoss  # 导入自定义的多任务联合损失计算器

In [8]:
def model_val(model, device, val_loader, criterion):
    model.eval()
    all_preds, all_labels = [], [] 
    total_val_loss_node = 0.0 # 分别统计两个 loss
    total_val_loss_edge = 0.0
    
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            n_out, e_out = model(batch)
            
            # 使用新逻辑：获取两个独立的 loss
            loss_n, loss_e,_ = criterion(n_out, batch.y_node, e_out, batch.y_edge)
            total_val_loss_node += loss_n.item()
            total_val_loss_edge += loss_e.item()
            
            # 【关键】：因为现在用了 BCEWithLogitsLoss，输出没有经过 sigmoid
            # 所以这里必须显式加上 sigmoid 转为概率
            all_preds.append(n_out.cpu()) 
            all_labels.append(batch.y_node.cpu())

    all_preds = torch.cat(all_preds, dim=0).numpy()
    all_labels = torch.cat(all_labels, dim=0).numpy()

    # 指标计算不变
    threshold = np.percentile(all_preds, 95) 
    preds = (all_preds > threshold).astype(int)
    all_labels_bin = (all_labels > 0.5).astype(int)
    
    auc = roc_auc_score(all_labels_bin, all_preds)
    acc = accuracy_score(all_labels_bin, preds)
    f1 = f1_score(all_labels_bin, preds, zero_division=0)
    
    model.train()
    return total_val_loss_node / len(val_loader), total_val_loss_edge / len(val_loader), auc, acc, f1

In [9]:
# def train_stress_flow(model_class, config):
#     device = torch.device(config['device'])
#     random.seed(config['seed']); np.random.seed(config['seed']); torch.manual_seed(config['seed'])
    
#     # 路径管理
#     cur_time = datetime.now().strftime("%Y_%m_%d")
#     save_path = os.path.join(config['save_root'], cur_time, model_class.__name__)
#     os.makedirs(save_path, exist_ok=True)

#     print(f"\n▶ Starting experiment: {model_class.__name__} on {config['device']}")
#     print("正在尝试加载数据...")
#     dataset = torch.load(config['data_path'], weights_only=False)
#     print(f"数据加载成功！共 {len(dataset)} 个案例。")

#     dataset = torch.load(config['data_path'], weights_only=False)
#     n_train = int(len(dataset) * config['train_size'])
#     train_loader = DataLoader(dataset[:n_train], batch_size=config['batch_size'], shuffle=True)
#     val_loader = DataLoader(dataset[n_train:], batch_size=config['batch_size'], shuffle=False)

#     model = model_class(input_dim=13, hidden_dim1=config['hidden_dim1'], hidden_dim2=config['hidden_dim2']).to(device)
#     optimizer = optim.Adam(model.parameters(), lr=config['lr'])
#     criterion = StressTaskLoss()

#     best_auc = 0.0


#     # 初始化记录文件
#     loss_file = os.path.join(save_path, 'losses.txt')
#     metric_file = os.path.join(save_path, 'val_metrics.csv')

    
#     # 1. 记录损失 (在 epoch 循环中)
#     # 假设损失文件改为记录两个 loss
#     with open(loss_file, 'a') as f:
#        f.write(f"{epoch+1},{avg_train_loss_n:.6f},{avg_train_loss_e:.6f}\n")

    
#     # 2. 记录指标 (metric_file)
#     with open(metric_file, 'a') as f:
#        f.write(f"{epoch+1},{val_loss_n:.6f},{val_loss_e:.6f},{auc:.6f},{acc:.6f},{f1:.6f}\n")

    
    


        
#     # 进度条只负责 Epoch
#     tbar = tqdm(range(config['epochs']), desc=f"Training {model_class.__name__}", bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {postfix}]')
    
#     for epoch in tbar:
#         model.train()
#         total_train_loss = 0.0
#         # 这里不再使用 tqdm，直接循环，界面会非常清爽
#         for batch in train_loader:
#             batch = batch.to(device)
#             optimizer.zero_grad()
#             n_out, e_out = model(batch)
            
#             #loss, loss_dict = criterion(n_out, batch.y_node, e_out, batch.y_edge)
#             #loss.backward()

#             # 直接获取两个损失
#             loss_n, loss_e = criterion(n_out, batch.y_node, e_out, batch.y_edge)
            
#             # 【这里是关键】：我们不再定义 total_loss 变量，直接对两项分别进行 backward
#             # 梯度在 PyTorch 中是累加的，这两行执行完，optimizer.step() 就会更新所有参数
#             loss_n.backward(retain_graph=True) # 计算 node 任务的梯度
#             loss_e.backward()                 # 计算 edge 任务的梯度
            
#             optimizer.step()
            
#             # 记录用于日志的 loss
#             total_train_loss_n += loss_n.item()
#             total_train_loss_e += loss_e.item()
        
#         avg_n, avg_e = train_n / len(train_loader), train_e / len(train_loader)
#         v_n, v_e, auc, acc, f1 = model_val(model, device, val_loader, criterion)

#         # 1. 记录 losses.txt
#         with open(loss_file, 'a') as f:
#             f.write(f"{epoch+1},{avg_train_loss:.6f}\n")
            
#         # 2. 记录 val_metrics.csv
#         with open(metric_file, 'a') as f:
#             f.write(f"{epoch+1},{val_loss:.6f},{auc:.6f},{acc:.6f},{f1:.6f}\n")

#         # 使用 set_postfix 更新指标，不要在循环里 print，否则会产生多行
#         tbar.set_postfix({
#             'loss_n': f"{avg_train_loss_n:.3f}",
#             'loss_e': f"{avg_train_loss_e:.3f}", 
#             'auc': f"{auc:.3f}", 
#             'acc': f"{acc:.4f}",
#             'f1': f"{f1:.3f}"
#         })
        
        
        
#         if auc > best_auc:
#             best_auc = auc
#             torch.save(model.state_dict(), os.path.join(save_path, 'best_model.pth'))

#         # 4. 定期保存 (每10个 epoch)
#         if (epoch + 1) % 10 == 0:
#             torch.save(model.state_dict(), os.path.join(save_path, f'model_epoch_{epoch+1}.pth'))


#     # 【就在这里】：在 for 循环完全结束之后调用 tbar.close()
#     tbar.close()

#     # 5. 训练结束保存 final_model.pth
#     torch.save(model.state_dict(), os.path.join(save_path, 'final_model.pth'))

def train_stress_flow(model_class, config):
    device = torch.device(config['device'])
    random.seed(config['seed']); np.random.seed(config['seed']); torch.manual_seed(config['seed'])
    
    # 路径管理
    cur_time = datetime.now().strftime("%Y_%m_%d")
    save_path = os.path.join(config['save_root'], cur_time, model_class.__name__)
    os.makedirs(save_path, exist_ok=True)

    print(f"\n▶ Starting experiment: {model_class.__name__} on {config['device']}")
    dataset = torch.load(config['data_path'], weights_only=False)
    n_train = int(len(dataset) * config['train_size'])
    train_loader = DataLoader(dataset[:n_train], batch_size=config['batch_size'], shuffle=True)
    val_loader = DataLoader(dataset[n_train:], batch_size=config['batch_size'], shuffle=False)

    model = model_class(input_dim=13, hidden_dim1=config['hidden_dim1'], hidden_dim2=config['hidden_dim2']).to(device)
    optimizer = optim.Adam(model.parameters(), lr=config['lr'])
    criterion = StressTaskLoss() # 确保这里与你导入的类名一致

    # 初始化记录文件路径
    loss_file = os.path.join(save_path, 'losses.txt')
    metric_file = os.path.join(save_path, 'val_metrics.csv')

    # 在循环前写入文件表头 (用 'w' 模式)
    with open(loss_file, 'w') as f: f.write("epoch,train_loss_n,train_loss_e\n")
    with open(metric_file, 'w') as f: f.write("epoch,val_loss_n,val_loss_e,auc,acc,f1\n")

    best_auc = 0.0
    tbar = tqdm(range(config['epochs']), desc=f"Training {model_class.__name__}")
    
    for epoch in tbar:
        model.train()
        total_train_n, total_train_e = 0.0, 0.0
        
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            n_out, e_out = model(batch)
            
            # 获取两个损失
            loss_n, loss_e,_ = criterion(n_out, batch.y_node, e_out, batch.y_edge)
            
            # 分别反向传播
            loss_n.backward(retain_graph=True)
            loss_e.backward()
            optimizer.step()
            
            total_train_n += loss_n.item()
            total_train_e += loss_e.item()
        
        avg_n = total_train_n / len(train_loader)
        avg_e = total_train_e / len(train_loader)
        
        # 获取验证结果
        v_n, v_e, auc, acc, f1 = model_val(model, device, val_loader, criterion)

        # 记录到文件 (使用 'a' 模式)
        with open(loss_file, 'a') as f:
            f.write(f"{epoch+1},{avg_n:.6f},{avg_e:.6f}\n")
        with open(metric_file, 'a') as f:
            f.write(f"{epoch+1},{v_n:.6f},{v_e:.6f},{auc:.6f},{acc:.6f},{f1:.6f}\n")

        tbar.set_postfix({'Ln': f"{avg_n:.3f}", 'Le': f"{avg_e:.3f}", 'auc': f"{auc:.3f}", 'f1': f"{f1:.3f}"})
        
        if auc > best_auc:
            best_auc = auc
            torch.save(model.state_dict(), os.path.join(save_path, 'best_model.pth'))

        if (epoch + 1) % 10 == 0:
            torch.save(model.state_dict(), os.path.join(save_path, f'model_epoch_{epoch+1}.pth'))

    tbar.close()
    torch.save(model.state_dict(), os.path.join(save_path, 'final_model.pth'))
    print(f"✅ 模型 {model_class.__name__} 训练完成。")


if __name__ == '__main__':
    config = {
        'hidden_dim1': 128,
        'hidden_dim2': 64,
        'batch_size': 2,
        'epochs': 60,
        'lr': 1e-4,
        'data_path': r'D:\2025-2026 RA\3-sitp 2026 基于图学习的楼板应力线找形\composite\graph_dataset\20260602_104cases.pt',
        'train_size': 0.8,
        'device': 'cuda',
        'save_root': './trained_model',
        'seed': 42
    }
    
    all_models = [PureSAGEModel, PureDynamicGATModel, PureGCNModel, PureClassicGATModel, ParallelHybridStressModel]
    for m in all_models:
        train_stress_flow(m, config)


▶ Starting experiment: PureSAGEModel on cuda


Training PureSAGEModel: 100%|█████████████| 60/60 [01:42<00:00,  1.71s/it, Ln=0.000, Le=0.004, auc=0.590, f1=0.000]


✅ 模型 PureSAGEModel 训练完成。

▶ Starting experiment: PureDynamicGATModel on cuda


Training PureDynamicGATModel: 100%|███████| 60/60 [03:10<00:00,  3.17s/it, Ln=0.000, Le=0.004, auc=0.577, f1=0.001]


✅ 模型 PureDynamicGATModel 训练完成。

▶ Starting experiment: PureGCNModel on cuda


...

✅ 模型 PureGCNModel 训练完成。

▶ Starting experiment: PureClassicGATModel on cuda


Training PureClassicGATModel: 100%|███████| 60/60 [01:47<00:00,  1.79s/it, Ln=0.000, Le=0.004, auc=0.527, f1=0.001]


✅ 模型 PureClassicGATModel 训练完成。

▶ Starting experiment: ParallelHybridStressModel on cuda


Training ParallelHybridStressModel: 100%|█| 60/60 [05:33<00:00,  5.56s/it, Ln=0.000, Le=0.004, auc=0.578, f1=0.001]

✅ 模型 ParallelHybridStressModel 训练完成。
